# Test finite steps, answer bias, and loud/quiet directions

The Python file contains the full implementation. This notebook launches a **separate background process**. Rerun the status cell manually to see progress; nothing continuously streams into the notebook.

New model experiments require your **original association checkpoints**, not only the measurement ZIP. Read `README.md` for definitions and interpretation. Included example plots are reanalyses of your existing data, not new OLMo results.

In [ ]:
from pathlib import Path
import importlib
import forgetting_mechanisms as fm

# Put this notebook beside forgetting_mechanisms.py.
# Keep your existing working OLMo/PyTorch environment.
print("Runner loaded:", Path(fm.__file__).name)

## Configure

Defaults target your original GPU experiment directory. Change paths for your machine. A new output directory is required after changing the design or code. Start with one seed/event for timing if desired; expand in a new output directory afterward.

In [ ]:
SOURCE = Path("/home/ubuntu/1/runs/olmo_association_v1")
OUTPUT = Path("/home/ubuntu/1/runs/forgetting_mechanisms_v1")
SETTINGS = fm.default_settings(SOURCE, OUTPUT)
SETTINGS.update(
    seeds=[1, 2, 3],
    events=[21, 47],                 # Wider schedule: [20, 21, 22, 46, 47, 48]
    device="cuda:0",
    threads=8,
    calibration_entities=8,          # Training prompts only; test does not fit the spectrum
    continuation_steps=16,
)
print("Planned cases:", len(SETTINGS["seeds"]) * len(SETTINGS["events"]))

## Launch once

This cell returns immediately. Logs and results go to `OUTPUT`. Re-running it while active is refused. Once interrupted or failed, running it again with identical settings skips completed cases and restarts unfinished ones.

In [ ]:
job = fm.launch(SETTINGS)
print(job["message"])
print("Output:", job["output"])

## Refresh status — rerun this cell whenever you want

A fresh heartbeat means the process is alive. Watch the phase, direction, dose or step to check whether work is advancing.

In [ ]:
fm.print_status(OUTPUT)

## Show available numbers and plots — rerun after cases finish

The default prints one numerical line per completed case and shows its plot. Set `FULL_REPORT=True` to print the entire intervention table. All numbers are saved to CSV/JSON regardless.

In [ ]:
FULL_REPORT = False
fm.show_results(OUTPUT, full=FULL_REPORT)

## Optional: read the latest worker log

Leave disabled during ordinary use. Enable if the status reports a failure or you want details.

In [ ]:
SHOW_LOG = False
if SHOW_LOG:
    logs = list(OUTPUT.glob("seed*/event*/worker.log"))
    path = max(logs, key=lambda p: p.stat().st_mtime) if logs else OUTPUT / "launcher.log"
    print("\n".join(path.read_text().splitlines()[-35:]) if path.exists() else "No log yet.")

## Optional: stop

Disabled so **Run All** does not cancel the job it just launched. Set `STOP=True` and run this cell only when you intend to stop.

In [ ]:
STOP = False
if STOP:
    print(fm.stop_run(OUTPUT))

## Optional: reanalyse your existing measurement ZIP

This requires no model checkpoint or GPU. It reproduces the existing dose and answer-bias numbers. It does not perform new causal interventions. The analysis cell stays quiet; the following cell displays results.

In [ ]:
ANALYZE_ARCHIVE = False
MEASUREMENT_ZIP = Path("/home/ubuntu/1/share_forgetting_measurement.zip")
ARCHIVE_OUTPUT = Path.cwd() / "archive_reanalysis"
if ANALYZE_ARCHIVE:
    _ = fm.analyze_measurements(MEASUREMENT_ZIP, ARCHIVE_OUTPUT)

In [ ]:
SHOW_ARCHIVE = False
if SHOW_ARCHIVE:
    fm.show_results(ARCHIVE_OUTPUT)

## Optional: local implementation test

The analytical tests are quick. The tiny-model end-to-end smoke run is available from the command line:

`python forgetting_mechanisms.py smoke --output /path/to/a/new/smoke_directory`

Its randomly initialized tiny model is an implementation check, **not evidence about OLMo forgetting**.

In [ ]:
RUN_CHECKS = False
if RUN_CHECKS:
    fm.self_test()